# 🏛️ Cuaderno 2 de 2 · M3 + M4 y evaluación con etiquetas

**Requisito:** haber corrido el **cuaderno 1** (M1 y M2). Este cuaderno **lee sus resultados de Drive**; no los recalcula.
**Qué hace:** saca `f_ssl` con **Sonata** (M3), una vez por variante (original, adaptado u otros pesos), corre el **Módulo 4** con cada una y **evalúa con las etiquetas** (mIoU).

**GPU:** cualquiera con 16 GB o más (la **L4** sirve). No tiene que ser la misma del cuaderno 1.

**¿M4 carga algo de M3 antes?** Sí: M4 necesita las tres cosas, y este cuaderno se las da en orden:
1. **M1** (el `.npy` de la escena) → lo trae la celda 3 desde Drive.
2. **M2** (`f_sem`) → lo trae la celda 8 desde Drive. **M3 no lo necesita**, así que puedes empezar este cuaderno **mientras el cuaderno 1 todavía corre M2**.
3. **M3** (`f_ssl`) → lo calcula la celda 7 y M4 la usa en la 8.

**Corre las celdas en orden, de la 1 a la 11** (la 12 es opcional). Si se cae la sesión, vuelve a correr desde la 1: lo ya guardado no se repite (`REHACER = True` lo fuerza).

| Celda | Qué hace |
|---|---|
| 1-2 | GPU, Drive, parámetros y repo |
| 3 | trae M1 de Drive (y M2 si ya está; si falta M1, te dice que corras el cuaderno 1) |
| 4-5 | entorno de Sonata: se **restaura de Drive**; si la GPU cambió, recompila solo |
| 6 | parche de memoria para Sonata (atención SDPA) |
| 7 | **M3**: `f_ssl` de cada variante |
| 8 | **M4** con cada variante |
| 9 | **evaluación** con etiquetas y todo guardado en Drive |
| 10 | **sonda lineal de Sonata** (linear probing) sobre cada `f_ssl` |
| 11 | **empaqueta lo necesario y lo descarga a tu laptop** (cuadros, comparativas, visor) |
| 12 | (opcional) control de la adaptación |

In [ ]:
# @title 1. GPU, Drive, carpetas y PARÁMETROS
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import os, torch
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob

DRIVE = "/content/drive/MyDrive/PFC1_HyperNCD_Sonata"
M1_ENT, M1_SAL = f"{DRIVE}/m1/entrada", f"{DRIVE}/m1/salida"
M2_ENT, M2_SAL = f"{DRIVE}/m2/entrada", f"{DRIVE}/m2/salida"
M3_ENTRENA, M3_PESOS, M3_SAL = f"{DRIVE}/m3/entrenamiento", f"{DRIVE}/m3/pesos", f"{DRIVE}/m3/salida"
M4_SAL   = f"{DRIVE}/m4/salida"
ENTORNOS = f"{DRIVE}/entornos"
for _d in (M1_ENT, M1_SAL, M2_ENT, M2_SAL, M3_ENTRENA, M3_PESOS, M3_SAL, M4_SAL, ENTORNOS):
    os.makedirs(_d, exist_ok=True)

def _mover(origen, destino):
    """Mueve origen -> destino solo si origen existe. Nunca pisa nada: si destino existe, no lo toca."""
    if not os.path.exists(origen):
        return
    if os.path.exists(destino):
        if os.path.isdir(origen) and os.path.isdir(destino):   # fusiona carpetas
            for x in os.listdir(origen):
                _mover(os.path.join(origen, x), os.path.join(destino, x))
            try:
                os.rmdir(origen)
            except OSError:
                pass
        else:
            print("  ya existe, no se toca:", destino.replace(DRIVE + "/", ""))
        return
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    shutil.move(origen, destino)
    print("  movido:", origen.replace(DRIVE + "/", ""), "->", destino.replace(DRIVE + "/", ""))

# M1: la salida (los .npy preprocesados, empaquetados)
_mover(f"{DRIVE}/data_m1.tar.gz",    f"{M1_SAL}/data_m1.tar.gz")
_mover(f"{DRIVE}/m1/data_m1.tar.gz", f"{M1_SAL}/data_m1.tar.gz")
# M2: checkpoints de entrada y resultados
_mover(f"{DRIVE}/datos_m2_descomprimidos",         f"{M2_ENT}/datos_m2_descomprimidos")
_mover(f"{DRIVE}/resultados_m2",                   f"{M2_SAL}/resultados_m2")
_mover(f"{DRIVE}/resultados_modulo2_hyperncd.zip", f"{M2_SAL}/resultados_modulo2_hyperncd.zip")
# Entornos compilados
_mover(f"{DRIVE}/hyperncd_env_cache.tar.gz", f"{ENTORNOS}/hyperncd_env_cache.tar.gz")
for _p in glob.glob(f"{DRIVE}/pointcept_env_*.tar.gz"):
    _mover(_p, f"{ENTORNOS}/{os.path.basename(_p)}")
# M3: si ya existía la carpeta anterior de la adaptación
_mover(f"{DRIVE}/sonata_adaptado/entrenamiento",       M3_ENTRENA)
_mover(f"{DRIVE}/sonata_adaptado/sonata_catedral.pth", f"{M3_PESOS}/sonata_catedral.pth")
_mover(f"{DRIVE}/sonata_adaptado/features",            M3_SAL)
print("Estructura lista en", DRIVE)


# ===================== PARÁMETROS (edita aquí) =====================
ESCENA = "capilla_etiquetada"                                  # la misma que usaste en el cuaderno 1
M3_VARIANTES = {                                               # nombre -> ruta de pesos (None = Sonata original de Meta)
    "original": None,
    "adaptado": f"{M3_PESOS}/sonata_catedral.pth",
    # "otro_modelo": "/content/drive/MyDrive/PFC1_HyperNCD_Sonata/m3/pesos/mis_pesos.pth",
}
K_LISTA = [6, 8, 12]                                           # grupos de M4 (6 = número de clases de la escena)
REHACER = False                                                # True = ignora lo guardado en Drive y recalcula
BRANCH = "main"
# ====================================================================
WORK = "/content/work"; os.makedirs(WORK, exist_ok=True)
OUT_DRIVE = f"{DRIVE}/m4/salida/{ESCENA}"; os.makedirs(OUT_DRIVE, exist_ok=True)
LOGS = f"{OUT_DRIVE}/logs"; os.makedirs(LOGS, exist_ok=True)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))

In [ ]:
# @title 2. Clonar / actualizar tu repo (rama indicada en BRANCH)
import os
from getpass import getpass

REPO_OWNER, REPO_NAME = "manyorycuevamendoza", "PFC1-HyperNCD-Sonata"
REPO_DIR = f"/content/{REPO_NAME}"

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if token:
        print("✅ Token leído desde Colab Secrets (GITHUB_TOKEN).")
except Exception:
    pass
if not token:
    token = getpass("Ingresa tu GitHub Personal Access Token (oculto): ").strip()

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin {BRANCH}
%cd {REPO_DIR}
!git log -1 --oneline
for f in ("experimentacion/evaluacion_etiquetada/evaluar_m4.py", "extraer_features.py", "experimentacion/modulo4_real/correr_modulo4_real.py"):
    assert os.path.exists(f"{REPO_DIR}/{f}"), f"Falta {f}: haz commit y push de ese archivo a la rama {BRANCH}."

In [ ]:
# @title 3. Traer M1 y M2 desde Drive (lo que dejó el cuaderno 1)
import os, shutil

M1D, M2D = f"{M1_SAL}/{ESCENA}", f"{M2_SAL}/{ESCENA}"
pedidos = {f"{M1D}/{ESCENA}.npy": f"{WORK}/{ESCENA}.npy",
           f"{M1D}/{ESCENA}.json": f"{WORK}/{ESCENA}.json",
           f"{M1D}/{ESCENA}_etiquetas.npy": f"{WORK}/{ESCENA}_etiquetas.npy",
           f"{M1D}/{ESCENA}_etiquetas.json": f"{WORK}/{ESCENA}_etiquetas.json"}
faltan = [os.path.relpath(o, DRIVE) for o in pedidos if not os.path.exists(o)]
assert not faltan, "Faltan estos archivos de M1 en Drive: " + ", ".join(faltan) + "\n→ Primero corre el cuaderno 1 (colab_1_m1_m2.ipynb) hasta su celda 5."
for o, d in pedidos.items():
    shutil.copy2(o, d)
NPY = f"{WORK}/{ESCENA}.npy"
import numpy as np
print("M1:", np.load(NPY, mmap_mode="r").shape)

# M2 es opcional aquí: M3 no lo necesita. Lo necesita M4 (celda 8), que lo vuelve a buscar en Drive.
FSEM = f"{WORK}/{ESCENA}_fsem.npz"
M2F = f"{M2D}/{ESCENA}_fsem.npz"
if os.path.exists(M2F):
    shutil.copy2(M2F, FSEM); print("M2: f_sem", np.load(FSEM)["feat"].shape)
else:
    print("⏳ M2 todavía no está en Drive (el cuaderno 1 puede seguir corriendo). M3 puede correr; M4 (celda 8) lo esperará.")

## Entorno de Sonata (se restaura de Drive)
La celda 4 busca `entornos/pointcept_env_<python>_<torch>_<cuda>_sm<GPU>.tar.gz` (un archivo **por tipo de GPU**). Si existe, lo descomprime (1-2 min) en vez de reinstalar; si no, lo construye (15-25 min) y lo guarda. La 5 comprueba que las extensiones compiladas funcionen **en la GPU de esta sesión** y, si no, las recompila y actualiza el archivo de Drive.

In [ ]:
# @title 4. Entorno de Sonata (caché en Drive, un archivo por tipo de GPU) (con caché en Drive: la 1ª vez compila, las siguientes restauran)
import os, sys, subprocess, torch

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if check and r.returncode != 0:
        raise RuntimeError(f"Falló (código {r.returncode}): {cmd}")

PYV  = f"py{sys.version_info.major}{sys.version_info.minor}"
TV   = torch.__version__.split("+")[0]
CU   = (torch.version.cuda or "cpu").replace(".", "")
CAP = "%d.%d" % torch.cuda.get_device_capability(0)
TAG  = f"{PYV}_torch{TV}_cu{CU}_sm{CAP.replace('.', '')}"      # un entorno por tipo de GPU
ENV_TAR = f"{ENTORNOS}/pointcept_env_{TAG}.tar.gz"
LEGACY = f"{ENTORNOS}/pointcept_env_{PYV}_torch{TV}_cu{CU}.tar.gz"   # nombre antiguo (sin GPU)
VENV = "/content/pcenv"
PYBIN = f"{VENV}/bin/python"
PIP   = f"{VENV}/bin/pip"
POINTCEPT = "/content/Pointcept"
CAP = "%d.%d" % torch.cuda.get_device_capability(0)

if not os.path.isdir(POINTCEPT):
    sh(f"git clone --depth 1 https://github.com/Pointcept/Pointcept {POINTCEPT}")

ORIGEN = ENV_TAR if os.path.exists(ENV_TAR) else (LEGACY if os.path.exists(LEGACY) else None)
if ORIGEN and not os.path.exists(PYBIN):
    print(f"⚡ Restaurando entorno desde Drive: {ORIGEN}")
    sh(f'tar -xzf "{ORIGEN}" -C /content')
elif not os.path.exists(PYBIN):
    print(f"🔧 Entorno {TAG} no está en Drive. Construyéndolo (15-25 min la primera vez)...")
    sh("pip install -q virtualenv")
    # --system-site-packages: reutiliza el torch de Colab (no se reinstala)
    sh(f"virtualenv -q --system-site-packages {VENV}")
    sh(f"{PIP} install -q setuptools wheel ninja")
    sh(f"{PIP} install -q addict yapf einops h5py plyfile termcolor timm tensorboardX tensorboard "
       f"wandb SharedArray open3d ftfy regex tqdm huggingface_hub scipy pyyaml packaging")
    if CU.startswith("11"):
        spc = "spconv-cu118"
    elif CU.startswith("12"):
        spc = "spconv-cu120"
    else:
        spc = "spconv-cu126"   # no hay ruedas de spconv para CUDA 13 (Colab nuevo); es un intento
        print(f"⚠️ CUDA {torch.version.cuda} y Python {sys.version.split()[0]} son más nuevos que lo que soporta spconv. "
              "Puede fallar; la comprobación de abajo dirá qué módulo.")
    sh(f"{PIP} install -q {spc}")
    # torch-scatter: rueda precompilada de PyG si existe; si no, se compila
    url = f"https://data.pyg.org/whl/torch-{TV}+cu{CU}.html"
    r = subprocess.run(f"{PIP} install -q torch-scatter --only-binary=:all: -f {url}", shell=True, executable="/bin/bash")
    if r.returncode != 0:
        print("No hay rueda de torch-scatter para esta versión de torch; compilando (~10 min)...")
        sh(f"TORCH_CUDA_ARCH_LIST='{CAP}' {PIP} install -q --no-build-isolation torch-scatter")
    # pointops (extensión CUDA de Pointcept)
    sh(f"cd {POINTCEPT}/libs/pointops && TORCH_CUDA_ARCH_LIST='{CAP}' {PIP} install -q --no-build-isolation .")
    # el resto de dependencias que importa pointcept.models / pointcept.datasets
    sh(f"{PIP} install -q peft transformers torch-geometric")
    # torch-cluster: no hay rueda para torch tan nuevo; se compila (~5-10 min)
    sh(f"TORCH_CUDA_ARCH_LIST='{CAP}' {PIP} install -q --no-build-isolation torch-cluster")
    _recien_construido = True

# pointgroup_ops (segmentacion de instancias) no se usa en el preentrenamiento de Sonata: modulo vacio
os.makedirs("/content/stubs", exist_ok=True)
open("/content/stubs/pointgroup_ops.py", "w").write("# stub: no se usa en Sonata")

# --- Comprobación módulo por módulo (así se ve CUÁL falla y por qué) ---
ENV = dict(os.environ, PYTHONPATH=f"{POINTCEPT}:/content/stubs")
MODULOS = ["torch", "spconv.pytorch", "torch_scatter", "pointops", "timm", "addict", "wandb",
           "tensorboardX", "open3d", "torch_cluster", "torch_geometric", "peft", "transformers", "pointcept.models", "pointcept.datasets"]
fallos = {}
for m in MODULOS:
    r = subprocess.run([PYBIN, "-c", f"import {m}"], cwd=POINTCEPT, env=ENV, capture_output=True, text=True)
    if r.returncode == 0:
        print(f"  ✅ {m}")
    else:
        ultima = (r.stderr.strip().splitlines() or ["(sin mensaje)"])[-1]
        fallos[m] = ultima
        print(f"  ❌ {m}: {ultima}")

if fallos:
    print("\nENTORNO INCOMPLETO. Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
    print("No se guardó en Drive. Si ya había una copia mala en entornos/, bórrala:", ENV_TAR)
    raise RuntimeError("Faltan módulos: " + ", ".join(fallos))

if globals().get("_recien_construido"):
    print("💾 Entorno OK. Guardando en Drive para la próxima vez...")
    sh(f'tar -czf "{ENV_TAR}" -C /content pcenv')
print("✅ Entorno listo.")

In [ ]:
# @title 5. Comprobar que el entorno funciona en ESTA GPU (si no, recompila y actualiza la caché en Drive)
import subprocess, os

PRUEBA = (
    "import torch, torch_scatter, pointops, torch_cluster\n"
    "x = torch.randn(8, device='cuda'); b = torch.tensor([0,0,1,1,2,2,3,3], device='cuda')\n"
    "torch_scatter.segment_coo(x, b, reduce='min'); torch.cuda.synchronize(); print('ok')\n"
)
open("/content/prueba_gpu.py", "w").write(PRUEBA)
def prueba():
    r = subprocess.run([PYBIN, "/content/prueba_gpu.py"], capture_output=True, text=True, env=ENV, cwd=POINTCEPT)
    return r.returncode == 0, (r.stderr.strip().splitlines() or [""])[-1]

ok, err = prueba()
if ok:
    print(f"✅ Entorno OK en {torch.cuda.get_device_name(0)} (capacidad {CAP}).")
else:
    print("⚠️ Las extensiones compiladas no corren en esta GPU:", err)
    print("   Recompilando para la capacidad", CAP, "(5-10 min). Después queda guardado en Drive.")
    ARCH = f"{CAP}+PTX"
    for pkg in ("torch-scatter", "torch-cluster"):
        sh(f"TORCH_CUDA_ARCH_LIST='{ARCH}' FORCE_CUDA=1 {PIP} install -q --no-build-isolation --no-deps --force-reinstall --no-cache-dir {pkg}")
    sh(f"cd {POINTCEPT}/libs/pointops && rm -rf build dist *.egg-info && TORCH_CUDA_ARCH_LIST='{ARCH}' FORCE_CUDA=1 "
       f"{PIP} install -q --no-build-isolation --no-deps --force-reinstall --no-cache-dir .")
    ok, err = prueba()
    assert ok, f"Sigue fallando tras recompilar: {err}"
    sh(f'tar -czf "{ENV_TAR}" -C /content pcenv')
    print("💾 Entorno recompilado y guardado en Drive para esta GPU.")

In [ ]:
# @title 6. M3 · parche de memoria (atención SDPA) en el Sonata del repo
import os, shutil, subprocess

# Sin flash-attn la atención densa de Sonata no cabe en memoria; SDPA es equivalente y usa mucho menos.
f = f"{REPO_DIR}/sonata/sonata/model.py"
src = open(f, encoding="utf-8").read()
viejo = """            attn = (q * self.scale) @ k.transpose(-2, -1)  # (N', H, K, K)
            if self.enable_rpe:
                attn = attn + self.rpe(self.get_rel_pos(point, order))
            if self.upcast_softmax:
                attn = attn.float()
            attn = self.softmax(attn)
            attn = self.attn_drop(attn).to(qkv.dtype)
            feat = (attn @ v).transpose(1, 2).reshape(-1, C)
"""
orig_ind = "".join(("    " + l) if l.strip() else l for l in viejo.splitlines(True))
nuevo = """            if (not self.enable_rpe) and (not self.upcast_softmax):
                feat = torch.nn.functional.scaled_dot_product_attention(
                    q, k, v.to(q.dtype),
                    dropout_p=self.attn_drop.p if self.training else 0.0,
                    scale=self.scale,
                )
                feat = feat.transpose(1, 2).reshape(-1, C).to(qkv.dtype)
            else:
""" + orig_ind
if "scaled_dot_product_attention" in src:
    print("Parche de atención ya aplicado.")
else:
    assert viejo in src, "No encontré el bloque de atención de Sonata (cambió el repo?)"
    open(f, "w", encoding="utf-8").write(src.replace(viejo, nuevo))
    print("✅ Parche de atención (SDPA) aplicado.")

In [ ]:
# @title 7. M3 · f_ssl con Sonata para cada variante de M3_VARIANTES (original, adaptado, ...)
import os, shutil, subprocess

import time
ENV2 = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/sonata")
for _n, _p in M3_VARIANTES.items():          # comprobar ANTES de gastar GPU que cada archivo de pesos existe
    if _p:
        assert os.path.exists(_p), f"No existe el archivo de pesos de '{_n}': {_p}"
        _st = os.stat(_p)
        print(f"Pesos '{_n}': {_p}  ({_st.st_size/2**20:.0f} MB, modificado {time.strftime('%Y-%m-%d %H:%M', time.localtime(_st.st_mtime))})")
    else:
        print(f"Pesos '{_n}': Sonata original de Meta (facebook/sonata, se baja de Hugging Face)")
FSSL = {}
for nombre, pesos in M3_VARIANTES.items():
    salida = f"{WORK}/{ESCENA}_feats_{nombre}.npz"
    cache = f"{DRIVE}/m3/salida/{ESCENA}/{ESCENA}_feats_{nombre}.npz"; os.makedirs(os.path.dirname(cache), exist_ok=True)
    c_pesos = cache.replace(".npz", ".pesos.txt")            # recuerda con qué pesos se calculó
    etiqueta = pesos or "original"
    mismo = os.path.exists(c_pesos) and open(c_pesos).read().strip() == etiqueta
    if os.path.exists(cache) and mismo and not REHACER:
        print(f"⚡ M3 '{nombre}' ya está en Drive con esos pesos; se copia.")
        shutil.copy2(cache, salida)
    else:
        print(f"\n=== M3 '{nombre}'  pesos: {pesos or 'Sonata original (Meta)'} ===", flush=True)
        cmd = [PYBIN, "extraer_features.py", "--input", NPY, "--output", salida, "--semilla", "0"] + (["--pesos", pesos] if pesos else [])
        r = subprocess.run(cmd, cwd=REPO_DIR, env=ENV2, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        print(r.stdout[-2500:]); open(f"{LOGS}/m3_{nombre}.log", "w").write(r.stdout)
        assert r.returncode == 0, f"Falló M3 '{nombre}' (mira el error de arriba)"
        shutil.copy2(salida, cache); open(c_pesos, "w").write(etiqueta)
    FSSL[nombre] = salida
print("\nVariantes de M3 listas:", list(FSSL))

In [ ]:
# @title 8. M4 · fusión y agrupamiento con cada variante de M3 (mismo M2, mismas regiones)
import os, shutil, subprocess

# M4 necesita M1 + M2 + M3. M2 lo deja el cuaderno 1 en Drive (puede haber terminado mientras corría M3).
M2F = f"{M2_SAL}/{ESCENA}/{ESCENA}_fsem.npz"
FSEM = f"{WORK}/{ESCENA}_fsem.npz"
assert os.path.exists(M2F), "M2 todavía no está en Drive: espera a que el cuaderno 1 llegue a su celda 8 y vuelve a correr esta celda."
shutil.copy2(M2F, FSEM)

M4_DIRS = {}
for nombre, fssl in FSSL.items():
    sal = f"{WORK}/m4_{nombre}"; M4_DIRS[nombre] = sal
    print(f"\n=== M4 con M3 '{nombre}' (M1 + M2 de Drive + M3 recién calculado) ===", flush=True)
    r = subprocess.run([PYBIN, f"{REPO_DIR}/experimentacion/modulo4_real/correr_modulo4_real.py",
                        "--npy", NPY, "--fsem", FSEM, "--fssl", fssl, "--salida", sal, "--k"] + [str(k) for k in K_LISTA],
                       cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-2000:]); open(f"{LOGS}/m4_{nombre}.log", "w").write(r.stdout)
    assert r.returncode == 0, f"Falló M4 '{nombre}' (mira el error de arriba)"
    shutil.copytree(sal, f"{OUT_DRIVE}/m4_{nombre}", dirs_exist_ok=True)      # a Drive YA: son los grupos por punto del visor
    print(f"💾 grupos de M4 '{nombre}' guardados en Drive")

In [ ]:
# @title 9. Evaluación con etiquetas (mIoU, ARI, NMI): con todas las clases y sin 'otro'; todo guardado en Drive
import os, shutil, subprocess
from IPython.display import Image, Markdown, display

grupos = [f"{n}={d}/grupos_PL13.npz" for n, d in M4_DIRS.items()]
IGNORAR_EN_SEGUNDA = ["otro"]      # "otro" = lo no segmentado (la nube completa); la 2ª evaluación no lo cuenta
for nombre_eval, extra in (("evaluacion", []), ("evaluacion_sin_otro", ["--ignorar", *IGNORAR_EN_SEGUNDA])):
    EVAL = f"{WORK}/{nombre_eval}"
    r = subprocess.run([PYBIN, f"{REPO_DIR}/experimentacion/evaluacion_etiquetada/evaluar_m4.py",
                        "--etiquetas", f"{WORK}/{ESCENA}_etiquetas.npy", "--info", f"{WORK}/{ESCENA}_etiquetas.json",
                        "--grupos", *grupos, "--salida", EVAL, *extra], cwd=REPO_DIR,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    assert r.returncode == 0, r.stdout[-3000:]
    display(Markdown(f"# ▶ {nombre_eval}"))
    display(Markdown(open(f"{EVAL}/evaluacion.md", encoding="utf-8").read()))
    display(Image(f"{EVAL}/evaluacion_miou.png"))
    shutil.copytree(EVAL, f"{OUT_DRIVE}/{nombre_eval}", dirs_exist_ok=True)
for n, d in M4_DIRS.items():
    shutil.copytree(d, f"{OUT_DRIVE}/m4_{n}", dirs_exist_ok=True)
print("\n💾 Resultados en Drive:", OUT_DRIVE)
!ls -lh "{OUT_DRIVE}"

In [ ]:
# @title 10. Sonda lineal de Sonata (linear probing) sobre cada f_ssl: el protocolo del paper de Sonata
# Entrena SOLO una capa lineal con etiquetas sobre f_ssl congelado y mide mIoU, mAcc y allAcc, con division ESPACIAL
# (train y test en bloques distintos). Mide el descriptor punto a punto, sin regiones ni M4. Tarda unos minutos por variante.
import os, shutil, subprocess

SONDA = f"{WORK}/sonda"
for nombre, fssl in FSSL.items():
    r = subprocess.run([PYBIN, f"{REPO_DIR}/experimentacion/evaluacion_etiquetada/sonda_lineal.py",
                        "--feats", fssl, "--npy", NPY, "--etiquetas", f"{WORK}/{ESCENA}_etiquetas.npy",
                        "--info", f"{WORK}/{ESCENA}_etiquetas.json", "--salida", SONDA, "--nombre", nombre],
                       cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-1800:]); open(f"{LOGS}/sonda_{nombre}.log", "w").write(r.stdout)
    assert r.returncode == 0, f"Falló la sonda de '{nombre}' (mira el error de arriba)"
shutil.copytree(SONDA, f"{OUT_DRIVE}/sonda_lineal", dirs_exist_ok=True)
print("💾 Sonda lineal guardada en", f"{OUT_DRIVE}/sonda_lineal")

In [ ]:
# @title 11. Empaquetar SOLO lo necesario para la laptop y descargarlo (cuadros, comparativas y visor)
import json, os, time, zipfile
from pathlib import Path

ESCENA = globals().get("ESCENA", "capilla_etiquetada")
DRIVE = globals().get("DRIVE", "/content/drive/MyDrive/PFC1_HyperNCD_Sonata")
CON_FEATURES = False        # True = incluye los f_ssl (~2,4 GB cada uno). Normalmente NO hace falta.
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

D = Path(DRIVE)
pedidos = []
def agregar(carpeta, filtro=None):
    base = D / carpeta
    if base.exists():
        for f in sorted(base.rglob("*")):
            if f.is_file() and (filtro is None or filtro(f)):
                pedidos.append((f, f.relative_to(D).as_posix()))
agregar(f"m1/salida/{ESCENA}")                                                  # M1: .npy, .json y etiquetas por punto
agregar(f"m1/entrada/{ESCENA}", lambda f: f.suffix == ".npz")                   # etiquetas crudas (el .las se deja fuera)
agregar(f"m2/salida/{ESCENA}")                                                  # M2: f_sem
agregar(f"m3/salida/{ESCENA}", (lambda f: True) if CON_FEATURES else (lambda f: f.suffix in (".txt", ".json", ".csv")))
agregar(f"m4/salida/{ESCENA}")                                                  # M4, evaluaciones (con FP/FN) y logs
assert pedidos, f"No hay nada de '{ESCENA}' en {DRIVE}: ¿corriste los cuadernos 1 y 2?"

salida = D / "paquetes" / f"paquete_{ESCENA}.zip"
salida.parent.mkdir(parents=True, exist_ok=True)
manifiesto = {"escena": ESCENA, "fecha": time.strftime("%Y-%m-%d %H:%M"), "con_features": CON_FEATURES,
              "archivos": [{"ruta": d, "MB": round(o.stat().st_size / 2**20, 2)} for o, d in pedidos]}
with zipfile.ZipFile(salida, "w", zipfile.ZIP_DEFLATED, compresslevel=3) as z:
    z.writestr("MANIFIESTO.json", json.dumps(manifiesto, indent=1, ensure_ascii=False))
    for i, (o, d) in enumerate(pedidos, 1):
        z.write(o, d); print(f"  [{i}/{len(pedidos)}] {d}  ({o.stat().st_size / 2**20:.1f} MB)")
print(f"\n✅ {salida}  ({salida.stat().st_size / 2**20:.0f} MB). Se descarga ahora a tu carpeta Descargas.")

from google.colab import files
files.download(str(salida))

In [ ]:
# @title 12. (Opcional) CONTROL de la adaptación: Sonata original y adaptado en el MISMO entorno, sobre PL13
# Mide cuánto cambia el descriptor SOLO por los pesos (mismo entorno, atención y semilla). Si el coseno es cercano a 1,
# la adaptación casi no movió el descriptor; si es bajo, el cambio es real y no viene del entorno.
HACER_CONTROL = False   # @param {type:"boolean"}
if HACER_CONTROL:
    import os, subprocess, numpy as np
    PL13_NPY = f"{REPO_DIR}/experimentacion/datos/modulo3_PL13/PL13_SCN0001.npy"
    assert os.path.exists(PL13_NPY), "Falta el .npy de PL13 en el repo clonado"
    ADAPT_PL13 = f"{DRIVE}/m3/salida/PL13_SCN0001_adaptado_feats.npz"
    assert os.path.exists(ADAPT_PL13), f"Falta {ADAPT_PL13} (el f_ssl adaptado de PL13 que ya sacaste)"
    ctrl = f"{WORK}/PL13_control_original.npz"
    r = subprocess.run([PYBIN, "extraer_features.py", "--input", PL13_NPY, "--output", ctrl, "--semilla", "0"],
                       cwd=REPO_DIR, env=ENV2, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-1500:]); assert r.returncode == 0
    a, b = np.load(ctrl), np.load(ADAPT_PL13)
    rng = np.random.default_rng(0); sel = np.sort(rng.choice(len(a["inverse"]), 200_000, replace=False))
    fa = a["feat"][a["inverse"][sel]].astype(np.float32); fb = b["feat"][b["inverse"][sel]].astype(np.float32)
    ca, cb = fa - fa.mean(0), fb - fb.mean(0)
    cos = (ca * cb).sum(1) / (np.linalg.norm(ca, axis=1) * np.linalg.norm(cb, axis=1) + 1e-8)
    print(f"Coseno centrado control(original) vs adaptado, mismo entorno: media {cos.mean():.3f}  (p5 {np.percentile(cos,5):.3f})")
    print("Compáralo con 0.54 (adaptado vs original de Khipu): si es parecido, el cambio viene de la adaptación y no del entorno.")
    shutil.copy2(ctrl, f"{DRIVE}/m3/salida/PL13_control_original_colab.npz")
else:
    print("Control omitido (HACER_CONTROL = False).")